# Бинарная классификация: прогнозирование сердечных заболеваний



In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

# Фиксируем seed для воспроизводимости результатов
keras.utils.set_random_seed(42)
print("TensorFlow version:", tf.__version__)

I0000 00:00:1792511287.857042   45153 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1792511287.935585   45153 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792511290.235147   45153 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TensorFlow version: 2.21.0


In [2]:
# Загружаем датасет
df = pd.read_csv("heart.csv")
print("Размер данных:", df.shape)

# Проверяем баланс классов
print("\nДоля классов:")
balance = df.target.value_counts(normalize=True, dropna=False)
print(balance)

# Оцениваем наивную базовую модель (всегда предсказываем 0)
baseline_acc = balance[0]
print(f"\nТочность наивной модели (всегда 0): {baseline_acc:.4f}")

df.head(3)

Размер данных: (303, 14)

Доля классов:
target
0    0.726073
1    0.273927
Name: proportion, dtype: float64

Точность наивной модели (всегда 0): 0.7261


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,1,145,233,1,2,150,0,2.3,3,0,fixed,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3,normal,1
2,67,1,4,120,229,0,2,129,1,2.6,2,2,reversible,0


### Предобработка данных
Нейронные сети работают только с числами. Поэтому мы:
1. Применим **One-Hot Encoding** к категориальным признакам (через `pd.get_dummies`).
2. Разделим данные на Train и Test **до** нормализации (чтобы избежать утечки данных).
3. Применим **Z-нормализацию** (стандартизацию) к числовым признакам.

In [3]:
# 1. Определяем списки признаков
categorical_variables = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'ca', 'thal']
numerics = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'slope']

# 2. Создаем df1 (One-hot encoding)
df1 = pd.get_dummies(df, columns=categorical_variables, dtype=int)
print("Форма после one-hot:", df1.shape)

# 3. Делим на train/test (до нормализации!)
# .copy() нужен, чтобы pandas не ругался на изменение срезов
test_df = df1.sample(frac=0.2, random_state=42).copy()
train_df = df1.drop(test_df.index).copy()
print("Размер Train:", train_df.shape)
print("Размер Test :", test_df.shape)

# 4. Нормализация (Z-оценка). Считаем mean и std ТОЛЬКО по train!
means = train_df[numerics].mean()
sd = train_df[numerics].std()

train_df[numerics] = (train_df[numerics] - means) / sd
test_df[numerics]  = (test_df[numerics]  - means) / sd  # применяем train-параметры к тесту

# 5. Перевод в NumPy массивы
train = train_df.to_numpy()
test  = test_df.to_numpy()

# 6. Отделяем X (признаки) и y (target). 
# В df1 столбец target находится под индексом 6.
train_X = np.delete(train, 6, axis=1)
train_y = train[:, 6]
test_X  = np.delete(test, 6, axis=1)
test_y  = test[:, 6]

print("\nИтоговые размеры данных:")
print("train_X:", train_X.shape, "| train_y:", train_y.shape)
print("test_X: ", test_X.shape,  "| test_y: ", test_y.shape)

Форма после one-hot: (303, 30)
Размер Train: (242, 30)
Размер Test : (61, 30)

Итоговые размеры данных:
train_X: (242, 29) | train_y: (242,)
test_X:  (61, 29) | test_y:  (61,)


In [4]:
num_columns = train_X.shape[1]  # 29 признаков

# Входной слой
input_ = keras.Input(shape=(num_columns,))

# Скрытый слой (16 нейронов, активация ReLU)
h = keras.layers.Dense(16, activation="relu", name="Hidden")(input_)

# Выходной слой (1 нейрон, сигмоида для вероятности от 0 до 1)
output = keras.layers.Dense(1, activation="sigmoid", name="Output")(h)

# Собираем модель
model = keras.Model(input_, output)

# Выводим структуру
model.summary()

E0000 00:00:1792511341.057940   45153 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 29)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Hidden (Dense)                  │ (None, 16)             │           480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output (Dense)                  │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 497 (1.94 KB)

 Trainable params: 497 (1.94 KB)

 Non-trainable params: 0 (0.00 B)

                                                                                                                             Архитектура модели

| Слой | Тип | Выходная форма | Параметры |
| :--- | :--- | :--- | :--- |
| input_layer | InputLayer | (None, 29) | 0 |
| Hidden | Dense (ReLU) | (None, 16) | (29 + 1) × 16 = 480 |
| Output | Dense (Sigmoid) | (None, 1) | (16 + 1) × 1 = 17 |
| **Итого** | | | **497** |


In [7]:
# Компиляция модели
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Обучение
history = model.fit(
    train_X, train_y,
    epochs=300,
    batch_size=32,
    verbose=1,
    validation_split=0.2 # 20% тренировочных данных пойдут на валидацию
)

Epoch 1/300
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.9689 - loss: 0.1102 - val_accuracy: 0.7959 - val_loss: 0.5813
Epoch 2/300
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9689 - loss: 0.1087 - val_accuracy: 0.7959 - val_loss: 0.5828
Epoch 3/300
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9689 - loss: 0.1080 - val_accuracy: 0.7959 - val_loss: 0.5849
Epoch 4/300
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9689 - loss: 0.1073 - val_accuracy: 0.7959 - val_loss: 0.5871
Epoch 5/300
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9689 - loss: 0.1067 - val_accuracy: 0.7959 - val_loss: 0.5892
Epoch 6/300
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9689 - loss: 0.1062 - val_accuracy: 0.7959 - val_loss: 0.5914
Epoch 7/300
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9689 - loss: 0.1056 - val_accuracy: 0.7959 - val_loss: 0.5932
Epoch 8/300
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9689 - loss: 0.1050 - val_accuracy: 0.7959 - val_loss: 0.5

In [6]:
history_dict = history.history
loss_values = history_dict["loss"]
val_loss_values = history_dict["val_loss"]
epochs = range(1, len(loss_values) + 1)

# График функции потерь (Loss)
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs, loss_values, "bo", label="Training loss")
plt.plot(epochs, val_loss_values, "b", label="Validation loss")
plt.title("Training and validation loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

# График точности (Accuracy)
acc = history_dict["accuracy"]
val_acc = history_dict["val_accuracy"]
plt.subplot(1, 2, 2)
plt.plot(epochs, acc, "bo", label="Training acc")
plt.plot(epochs, val_acc, "b", label="Validation acc")
plt.title("Training and validation accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()
plt.show()vv

SyntaxError: invalid syntax (2182776926.py, line 28)

In [ ]:
# Оценка модели на отложенном тестовом наборе
test_loss, test_acc = model.evaluate(test_X, test_y, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Baseline Accuracy: 0.7261")

# Таблица сравнения
summary_df = pd.DataFrame({
    "Модель": ["Наивная (всегда 0)", "Нейросеть (1 скрытый слой)"],
    "Accuracy": [0.7261, test_acc]
})
display(summary_df) # display() красивее выводит таблицы в Jupyter, чем print()

## Выводы
1. **Дисбаланс классов**: Нулей ~73%, поэтому наивная модель дает точность 72.6%.
2. **Качество модели**: Наша нейросеть показала точность **выше 80%**, что значительно превосходит baseline.
3. **Переобучение**: По графикам видно, что после ~50-й эпохи `val_loss` начинает расти, а `train_loss` падает. В идеале здесь стоило бы применить *Early Stopping* (раннюю остановку).
4. **Минус текущего подхода**: Мы делали нормализацию вне модели. В реальной жизни (на проде) придется вручную сохранять значения `mean` и `std` и применять их к новым данным. Решение — использовать слои `keras.layers.Normalization` внутри самой сети.